In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%cd drive/My Drive/FoodDB_NDB_matching

/content/drive/My Drive/FoodDB_NDB_matching


In [ ]:
import random as rand
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from wordcloud import WordCloud,STOPWORDS
import json
import json
import sklearn
import itertools
import pydotplus
import seaborn as sns
import re
import ast
import time
import copy
import joblib
import re
import os
from os import path

In [ ]:
!pip install openpyxl
import openpyxl

In [ ]:
fdb_df = pd.read_csv('products_with_categories.csv')
NDB_df = pd.read_csv('NDB_matching_data_FINAL.csv')

<ipython-input-6-9c641b1e22be>:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  fdb_df = pd.read_csv('products_with_categories.csv')


# Filter out food groups

In [ ]:
# Unavailable due to legal restrictions on access to underlying foodDB data. See Data Availability Statement in accompanying paper.
fdb_df = pd.read_csv('products_with_categories.csv')
NDB_df = pd.read_csv('NDB_matching_data_FINAL.csv')

<ipython-input-23-9c641b1e22be>:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  fdb_df = pd.read_csv('products_with_categories.csv')


In [ ]:
len(fdb_df['product_list_name'].unique())

75060

In [ ]:
len(fdb_df[fdb_df['aisle']=='Chocolate']['product_list_name'].unique())

1764

In [ ]:
ndb_cat = NDB_df['NDNS Food group'].unique().tolist()

In [ ]:
aisles_list = fdb_df['aisle'].unique().tolist()

In [ ]:
len(aisles)

2870

In [ ]:
aisle_length = []
for i in aisles_list:
  aisle_length.append(len(fdb_df[fdb_df['aisle']==i]))

# Matching with GPT-4 LLM

These scripts contain code used for matching foodDB items to the NDB items using the GPT-4 LLM. As this analysis was conducted in 2023 the syntax of the openAI package used in this analysis is no longer in use and will have to be updated for any future analyses.   

In [ ]:
!pip install openai

In [ ]:
import openai

In [ ]:
MODEL = "gpt-4"
SYSTEM_ROLE = "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly the composition of food products in the UK"

In [ ]:
print(response['choices'][0]['message']['content'])

In the heart of the UK, where the flavours are vast,
A question arises, "Is this a test prompt?" asked.
From scones to the pasties, and tea that's well-brewed,
Each recipe a story, each dish a mood.

Fish and chips by the seaside, a joy so profound,
In the bustling city, curry's aroma is found.
From the Scottish haggis, to the Welsh rarebit zest,
"Is this a test prompt?" Yes, it's a culinary quest.


## Matching food categories

In [ ]:
# Prompt to match food categories in NDB to aisles in foodDB
def matching_prompt_aisles(LIST1, LIST2):
  matching_prompt = f"""
Create a dictionary with keys consisting of items in LIST_NDB = {LIST1} which map to a list of appropriate items in the list LIST_AISLE={LIST2} based on the following criteria:
1) Include matches in LIST_AISLE on the basis of whether the food category in LIST_NDB is likely to be found in the corresponinding aisle of a UK supermarket. E.g.For 'ICE CREAM', consider aisles like 'Desserts' and 'Frozen Party Food' as appropriate matches.
2) If there a multiple potential matches include them all in the mapping list.
3) If there are no appropriate matches, match to an empty list.
4) Don't make any items up.
5) Please match categories based on where the food items would most likely be stored or displayed in a supermarket, considering temperature requirements, occasion, and type of food.
6) Please double-check each category for appropriate aisle matches, especially considering the type of storage or presentation of the item (e.g., frozen, dried)
7) If there is even a small possibility the aisle could contain the food category include it in the list.
Follow these instructions and return a dictionary mapping the items in LIST_NDB to the list matches in LIST_AISLE:
Just print the mapping dictionary, please don't write any code or text.
"""
  return matching_prompt

In [ ]:
PROMPT = matching_prompt_aisles(LIST1=ndb_cat, LIST2=aisles_list[:200])

In [ ]:
PROMPT_TEST = matching_prompt_aisles(LIST1=ndb_cat, LIST2=aisles_list[:200])

In [ ]:
def categorise_fdb(list_categories):

  SYSTEM_ROLE_AISLES = "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly the composition of food products and layouts of UK supermarkets"

  with open('NDB_CAT_TO_AISLE.json', 'r') as json_file:
    data_dict_final = json.load(json_file)

  categories_sub_list = split_list(list_categories, N=20)

  sub_sub_lists = []
  for i in categories_sub_list:
    split_lists_tail = split_list(i, N=2)
    for j in split_lists_tail:
       sub_sub_lists.append(j)

  count=0

  for sub_list in sub_sub_lists:
  #for sub_list in categories_sub_list:
    PROMPT = matching_prompt_aisles(LIST1=ndb_cat, LIST2=sub_list)

    response = openai.ChatCompletion.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": SYSTEM_ROLE_AISLES},
          {"role": "user", "content": f"{PROMPT}"},
      ],
      temperature=0,
    )

    mapping_str_response = response['choices'][0]['message']['content']
    match = re.search(r"\{.*?\}", mapping_str_response, re.DOTALL)
    if match:
      data_str = match.group(0)
      data_dict = ast.literal_eval(data_str)
      print(data_dict)

      for ndb_category, potential_matches in data_dict.items():
        if data_dict_final[ndb_category] is not []:
          potential_matches = potential_matches if potential_matches is not None else []
          union_list = list(set(data_dict_final[ndb_category]) | set(potential_matches))
          data_dict_final[ndb_category] = union_list
        else:
          data_dict_final[ndb_category] = potential_matches

      save_mapping(mapping_dictionary=data_dict_final, file_name='NDB_CAT_TO_AISLE')
      count+=1
      print(f'finished prompt {count}')

    else:
        print("GPT4 has not returned a dictionary. The response was: ")
        print(mapping_str_response)
        raise ValueError('No dictionary found!')


  return data_dict_final




In [ ]:
sub_sub_lists[-1]

In [ ]:
SYSTEM_ROLE_AISLES = "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly the composition of food products and layouts of UK supermarkets"
PROMPT = matching_prompt_aisles(LIST1=ndb_cat, LIST2=sub_sub_lists[-1])

response = openai.ChatCompletion.create(
  model=MODEL,
  messages=[
      {"role": "system", "content": SYSTEM_ROLE_AISLES},
      {"role": "user", "content": f"{PROMPT}"},
  ],
  temperature=0,
)

mapping_str_response = response['choices'][0]['message']['content']
match = re.search(r"\{.*?\}", mapping_str_response, re.DOTALL)
if match:
  data_str = match.group(0)
  data_dict = ast.literal_eval(data_str)

In [ ]:
with open('NDB_CAT_TO_AISLE.json', 'r') as json_file:
    data_dict_final = json.load(json_file)

for ndb_category, potential_matches in data_dict.items():
  if data_dict_final[ndb_category] is not []:
    potential_matches = potential_matches if potential_matches is not None else []
    union_list = list(set(data_dict_final[ndb_category]) | set(potential_matches))
    data_dict_final[ndb_category] = union_list
  else:
    data_dict_final[ndb_category] = potential_matches

  save_mapping(mapping_dictionary=data_dict_final, file_name='NDB_CAT_TO_AISLE')

In [ ]:
categories_sub_list = split_list(aisles_list[200:], N=20)

sub_sub_lists = []
for i in categories_sub_list[16:]:
  split_lists_tail = split_list(i, N=2)
  for j in split_lists_tail:
    sub_sub_lists.append(j)

In [ ]:
categories_sub_list = split_list(aisles_list[200:], N=20)

In [ ]:
for i in categories_sub_list[16]:
  if 'Pulses, Lentils & Beans' in i:
    print(i)

Pulses, Lentils & Beans


In [ ]:
food_db_cat_dictionary = categorise_fdb(list_categories=aisles_list[200:])

In [ ]:
with open('NDB_CAT_TO_AISLE.json', 'w') as json_file:
  json.dump(data_dict, json_file, indent=4)

## Matching food items

The TikToken package estimates the number of tokens per prompt. The cap was set at 10000

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4")

In [ ]:
## test to get an idea of tokens per prompt
len(encoding.encode("chair one "))

3

### macthing prompts

In [ ]:
def matching_prompt_default(LIST1, LIST2):
  matching_prompt = f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of food products and composition rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Do not match a single items (e.g.'Potato') with a composite dish containing that item (e.g. a potato soup).
Consider ingredients as well as food group, e.g. do not match beef soup to chicken soup even though they are both soups.
3)If there a multiple potential matches include them all in the mapping list.
4)Ignore minor differences in seasonings or flavourings.
5)Do not make up any key or matched items. Check each match follows these instructions. Don't print your reasoning.Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

NDB_pasta_pizza = ['PIZZA', 'PASTA MANUFACTURED PRODUCTS & READY MEALS']

### pasta/pizza ###
#stop_words = []

def matching_prompt_pizza(LIST1, LIST2):
  matching_prompt = f"""
Go through each pizza/pasta item in the list LIST_FDB={LIST1} and determine if a similar pizza/pasta item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list of matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of the similarity of food products, rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Ensure the recipes/pizza type closely match. E.g. match spagetti carbonara in LIST_SHES with any spagetti carbonara in LIST_FDB, not with spagetti bolognese.
Strictly ensure that if the item in LIST_SHES is vegetarian, only vegetarian items from LIST_FDB are included.
3)If there a multiple potential matches include them all in the mapping list.
4)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
NDB_rice = ['OTHER RICE INCLUDING HOMEMADE DISHES', 'RICE MANUFACTURED PRODUCTS & READY MEALS']

### pasta/pizza ###
#stop_words = []

def matching_prompt_rice(LIST1, LIST2):
  matching_prompt = f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list of matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of the similarity of food products, rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Ensure the recipes type closely match. E.g. match a mushroom risotto in LIST_SHES with any mushroom risotto in LIST_FDB, not with a chicken risotto.
Strictly ensure that if the item in LIST_SHES is vegetarian, only vegetarian items from LIST_FDB are included.
Strictly ensure that the rice type is consistent, i.e. only include brown rice items from LIST_FDB in the list if the item in LIST_SHES is a brown rice.
3)If there a multiple potential matches include them all in the mapping list.
4)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
def matching_prompt_PB_alt(LIST1, LIST2):
  matching_prompt = f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list of matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of the similarity of food products, rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Do not match single ingredient items (e.g.tofu) with multi-ingredient products containing that item (e.g. a tofu stir fry).
However, if the item in LIST_SHES does contain multiple ingredients, ensure the recipes closely match. E.g. match a quorn lasagne with lasagnes containing quorn, not with vegetable or lamb lasagnes.
Strictly ensure no meat items are included in the matched lists, only closely matching meat-alternatives (e.g. vegan chicken, tofu, tempe etc)
3)If there a multiple potential matches include them all in the mapping list.
4)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
### red meat ###
stop_words_red_meat = ['plant-based', 'vegan', 'veggie', 'chicken', 'turkey']

def matching_prompt_red_meat(LIST1, LIST2):
  matching_prompt = f"""
Go through each meat item in the list LIST_FDB={LIST1} and determine if a similar meat item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list of matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of the similarity of food products, rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Do not match single ingredient items (e.g.roast lamb) with multi-ingredient products containing that item (e.g. a lamb curry).
However, if the item in LIST_SHES does contain multiple ingredients, ensure the recipes closely match. E.g. match lamb tikka masala with tikka masala products containing lamb, not with lamb balti or lamb lasagne.
Strictly ensure the meat type in the list is the same as in LIST_SHES. E.g if the item in LIST_SHES is a beef sausage, only include items from LIST_FDB that are sausages made of beef in the list, not pork sausages.
3)If there a multiple potential matches include them all in the mapping list.
4)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
## Poultry ##

def matching_prompt_poultry(LIST1, LIST2):
  matching_prompt = f"""
Go through each poultry item in the list LIST_FDB={LIST1} and determine if a similar poultry item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of the similarity of poultry products, rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Do not match a single items (e.g.'Chicken breast') with a composite dish containing that item (e.g. a chicken curry).
However, if the item is a composite item ensure the recipes closely match, e.g. only match chicken stir fires with stir fry products containing chicken, not with chicken breast alone.
Strictly ensure the poultry product in the list is the same as in LIST_SHES. E.g if the item in LIST_SHES is a turkey product, only include items that contain turkey in the matched list, not chicken.
Consider the cut of meat; e.g. only match chicken legs to chicken legs and chicken breasts to chicken breasts.
Only consider items in LIST_FDB that contain poultry when assigning matches.
3)If there a multiple potential matches include them all in the mapping list.
4)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
### seafood ##

def matching_prompt_seafood(LIST1, LIST2):
  matching_prompt = f"""
Go through each food item in the list LIST_FDB={LIST1} and determine if there is a similar seafood item in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to a list of matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of food products, rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Do not match single ingredient items (e.g smoked salmon) with a multi-ingredient item containing that single ingredien (e.g. a smoked salmon sandwhich).
However, if the item in LIST_SHES contains multiple ingredients, ensure the recipes of the matched item in LIST_FDB closely match.
For example, only include seafood pasta dishes from LIST_FDB if the item in LIST_SHES is a seafood pasta, do not include individual seafood items.
Strictly ensure every seafood item in the list is the same as in LIST_SHES. E.g if the item in LIST_SHES is a mackerel product, only include products containing mackerel in the list, not salmon products.
Only consider seafood items when assigning matches.
3)If there a multiple potential matches include them all in the mapping list.
4)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
############# DAIRY/DAIRY ALT ITEMS ##############

def matching_prompt_dairy(LIST1, LIST2):
  matching_prompt = f"""
Go through each food item in the list LIST_FDB={LIST1} and determine if there is a similar dairy/dairy alternative item in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to a list of matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of food products, rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Do not match single ingredient items (e.g halloumi) with a multi-ingredient item containing that single ingredient (e.g. a halloumi burger).
Strictly ensure every dairy/dairy alternative item included in the list is the same dairy/dairy alternative as in LIST_SHES.
E.g if the item in LIST_SHES is a yoghurt, only include yoghurt products in the list, no ice creams.
Do not include any non-dairy/dairy alternative items from LIST_FDB in the list (e.g a vegan ice cream) if the item in LIST_SHES contains dairy and vice versa.
3)If there a multiple potential matches include them all in the mapping list.
4)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
#cereals
def matching_prompt_cereals(LIST1, LIST2):
  matching_prompt = f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of food products and composition rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
3)Ensure the cereal type included in the list is the same as that in LIST_SHES, i.e. if the item in LIST_SHES is a muesli with fruit only, only include mueslies in the list that only have fruit and so on.
Ensure each cereal type (porridge/muesli etc) is consistent between the key and the matched items; i.e if the item in LIST_SHES is porridge, only include appropriate porridge items in the matched list.
4)Consider the additional info from items in LIST_SHES, such as if the item is gluten free/low sugar, only include appropriate gluten free/low sugar items in the matching list respectively.
5)If there a multiple potential matches include them all in the mapping list.
6)Ignore minor differences in seasonings or flavourings.
7)Do not make up any key or matched items.
Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt


In [ ]:
NDB_cat_vegetables = ['NUTS AND SEEDS', 'OTHER MANUFACTURED VEGETABLE PRODUCTS INCL RM', 'LEAFY GREEN VEGETABLES NOT RAW', 'OTHER POTATOES INCLUDING HOMEMADE DISHES',
                      'APPLES AND PEARS NOT CANNED', 'CANNED FRUIT IN SYRUP', 'PEAS NOT RAW' ]

#vegetables
def matching_prompt_vegetables(LIST1, LIST2):
  matching_prompt = f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar food item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list of matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of the similarity of food products, rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Do not match single ingredient items (e.g. a potato) with multi-ingredient products containing that item (e.g. a potato curry).
However, if the item in LIST_SHES does contain multiple ingredients, ensure the recipes closely match. E.g. match vegetable curries with vegetable curries of a similar recipe and ingredient content.
Strictly ensure the vegetable type in the list is the same as in LIST_SHES. E.g if the item in LIST_SHES is hazlenuts, only include items from LIST_FDB that are hazelnuts.
Do not include any meat containing items in the matching list.
3)If there a multiple potential matches include them all in the mapping list.
4)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
NDB_cat_bread = ['WHOLEMEAL BREAD',
'WHITE BREAD (NOT HIGH FIBRE; NOT MULTISEED BREAD)',
'BROWN GRANARY AND WHEATGERM BREAD',
'OTHER BREAD']

#bread
def matching_prompt_bread(LIST1, LIST2):
  matching_prompt = f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of food products and composition rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
3)Ensure the bread type included in the list is the same as that in LIST_SHES, i.e. if the item in LIST_SHES is a muesli with fruit only, only include mueslies in the list that only have fruit and so on.
Ensure each bread type (naan/sourdough etc) is consistent between the key and the matched items; i.e if the item in LIST_SHES is a wholemeal bread, only include appropriate wholemeal bread items in the matched list.
4)Consider the additional info from items in LIST_SHES, such as if the item is gluten free/multiseed, only include appropriate gluten free/multiseed items in the matching list respectively.
5)If there a multiple potential matches include them all in the mapping list.
6)Ignore minor differences in seasonings or flavourings.
7)Do not make up any key or matched items.
Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
# confectionary/sweet snacks
NDB_sweets_snacks = ['CRISPS AND SAVOURY SNACKS', 'SUGAR CONFECTIONERY', 'SWEET SPREADS FILLINGS AND ICING','SUGAR', 'BUNS CAKES & PASTRIES HOMEMADE']

def matching_prompt_confectionary(LIST1, LIST2):
  matching_prompt=f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of food products and composition rather than partial matches within a string.
For example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Ensure the food type in the list is the same as that in the key, i.e. if the key refers to a salt and vinegar crisps, only inlcude salt and vinegar crisps in the matched list, not cheese and onion crisps.
Ensure cake/pastry/bun type is consistent. For example if the item in LIST_SHES is a cinammon pastry, only include items from LIST_FDB that are pastries with cinammon.
3)If there a multiple potential matches include them all in the mapping list.
4)Ignore minor differences in seasonings or flavourings.
5)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt


In [ ]:
NDB_drinks = ['COFFEE (MADE-UP WEIGHT)', 'SOFT DRINKS NOT LOW CALORIE CARBONATED',
               'SOFT DRINKS LOW CALORIE CONCENTRATED', 'BEVERAGES DRY WEIGHT']

def matching_prompt_drinks(LIST1, LIST2):
  matching_prompt=f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar drink item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of drink products and composition rather than partial matches within a string.
To give a food example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Ensure the drink type in the list is the same as that in the key, i.e. if the key refers to a cappuccino, only include cappuccinos in the matched list, not lattes.
Consider additional info given in LIST_SHES when assigning matches, e.g. if the drink does not have added sugar or is decaf, only include items from LIST_FDB that do not have added sugar or are decaf respectively.
3)If there a multiple potential matches include them all in the mapping list.
4)Ignore minor differences in seasonings or flavourings.
5)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt


In [ ]:
def matching_prompt_eggs(LIST1, LIST2):
  matching_prompt=f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of food products and composition rather than partial matches within a string.
To give an example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Ensure the food type in the list of items from LIST_FDB is the same as that in the key from LIST_SHES, i.e. if the item in LIST_SHES refers to a duck eggs, only include duck eggs in the matched list, not quail or chcken eggs.
3)If there a multiple potential matches include them all in the mapping list.
4)Ignore minor differences in seasonings or flavourings.
5)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

In [ ]:
NDB_soups = ["SOUP HOMEMADE",
"SOUP MANUFACTURED/ RETAIL" ]

def matching_prompt_soups(LIST1, LIST2):
  matching_prompt=f"""
Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
1)If there are no appropriate matches, match to [].
2)Matches should only be considered based on your knowledge of food products and composition rather than partial matches within a string.
To give an example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
Ensure the food type in the list of items from LIST_FDB is the same as that in the key from LIST_SHES, i.e. if the item in LIST_SHES refers to a vegetable soup, only include vegetable soups, not chicken and vegetable soups.
3)If there a multiple potential matches include them all in the mapping list.
4)Ignore minor differences in seasonings or flavourings.
5)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
"""
  return matching_prompt

# def matching_prompt_soups(LIST1, LIST2):
#   matching_prompt=f"""
# Go through each item in the list LIST_FDB={LIST1} and determine if a similar item exists in the list LIST_SHES={LIST2}.
# Follow these instructions and return a dictionary mapping the items in LIST_SHES to the list matches in LIST_FDB:
# 1)If there are no appropriate matches, match to [].
# 2)Matches should only be considered based on your knowledge of soup products and composition rather than partial matches within a string.
# To give a food example, Aubergine and Eggplant refer to the same food, but a string match would not match them.
# Ensure the recipes of the soups included from LIST_FDB in the matched list are similar to those within the key, i.e. if the key refers to a vegetable soup,include only vegetable based soups in the matched list,
# do not inlcude no soups containing only meat. Conisder whether the soup in LIST_SHES is vegetarian or not when assigining matches.
# 3)If there a multiple potential matches include them all in the mapping list.
# 4)Ignore minor differences in seasonings or flavourings.
# 5)Do not make up any key or matched items. Double check each match follows these instructions. Don't print your reasoning. Only print the mapping dictionary, do not write any code or text.
# """
#   return matching_prompt




### prompt_dictionary

In [ ]:
prompt_dict = {

          'default': {
        'prompt': matching_prompt_default,
        'stop_words': [],
        'SYSTEM_ROLE': "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly food composition/recipes"

    },

    'red_meat': {
        'prompt': matching_prompt_red_meat,
        'stop_words': ['plant-based', 'vegan', 'veggie'],
        'SYSTEM_ROLE': "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly the difference between meat products"

    },
    'pizza_pasta': {
        'prompt': matching_prompt_pizza,
        'stop_words': []
    },
    'seafood': {
        'prompt': matching_prompt_seafood,
        'stop_words': [],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly seafood products"
    },
    'poultry': {
        'prompt': matching_prompt_poultry,
        'stop_words': []
    },
    'cereals': {
        'prompt': matching_prompt_cereals,
        'stop_words': ['falafal', 'cake', 'dessert', 'ice cream', 'tub', 'pudding', 'soup', 'cheese', 'crisps', 'sausage', 'beef', 'bacon', 'pork', 'gammon','lamb', 'chicken', 'turkey'],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly cereal products"
    }
    ,
    'dairy': {
        'prompt': matching_prompt_dairy,
        'stop_words': [],
        'SYSTEM_ROLE': "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly dairy and dairy alternative products/food items"
    },
    'vegetables_fruit':{
        'prompt': matching_prompt_vegetables,
        'stop_words': ['dessert', 'ice cream', 'tub'],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food available in the United Kingdom, particularly fruits, vegetables, nuts and seeds"
    },
    'bread':{
        'prompt': matching_prompt_bread,
        'stop_words': [],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly the difference between bread products"
    },
    'confectionary':{
        'prompt': matching_prompt_confectionary,
        'stop_words': [],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly the difference between cake/snack/confectionary products"
    }
    ,
    'drinks':{
        'prompt': matching_prompt_drinks,
        'stop_words': [],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly the beverages that are consumed/purchased"
    }
     ,
    'eggs':{
        'prompt': matching_prompt_eggs,
        'stop_words': ['sardine','fillet','salmon', 'tuna' ,'falafal', 'cake', 'dessert',
              'ice cream', 'tub', 'pudding', 'soup', 'crisps', 'sausage', 'beef',
              'bacon', 'pork', 'gammon','lamb', 'chicken', 'turkey',
              'yoghurt', 'yogurt', 'drink', 'chocolate', 'sauce','cod', 'milk', 'rice', 'pizza'],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly egg products"
    }
     ,
    'soups':{
        'prompt': matching_prompt_default,
        'stop_words': [],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly soups"
    },
    'rice': {'prompt': matching_prompt_rice,
        'stop_words': ["pizza", "pasta"],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly rice based products/meals" }
    ,
    'PB_alt': {'prompt': matching_prompt_PB_alt,
        'stop_words':  ['salmon', 'tuna', 'beef','sardine', 'anchovy', 'mackerel'],
        'SYSTEM_ROLE' : "You are a helpful researcher with an extensive knowledge of food in the United Kingdom, particularly particularly the composition/recipes of available food products" }



}


In [ ]:
stop_words = ['salmon', 'tuna', 'beef','sardine', 'anchovy', 'mackerel']

## Code to run the GPT matching

LIST1 corresponds to the list we want to match to, e.g. FoodDB, FSA recipe database etc
LIST2 corresponds to the items that we already have, in our case SHeS and NDB.

In [ ]:
# Split a list into N approximately equally sized sublists -> For breaking down the longer list of FSA/FoodDB items
def split_list(lst, N):
    k, m = divmod(len(lst), N)
    return [lst[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(N)]

In [ ]:
## Breaks down the longer list into a number of sublists and returns a list of prompts such that the token count of each does not exceed the cap.
def prep_prompt(LIST1, LIST2, max_tokens, prompt_label):

    prompt_list = []

    PROMPT = prompt_dict[prompt_label]['prompt'](LIST1, LIST2)

    #PROMPT = matching_prompt_3(LIST1, LIST2)
    #print(type(PROMPT))
    num_tokens = len(encoding.encode(PROMPT))
    print(f"starting token: {num_tokens}")

    if num_tokens < max_tokens:
        prompt_list.append(PROMPT)
    else:
        N = 2
        while True:
            split_list_long = split_list(LIST1, N)
            all_prompts_valid = True

            for sub_list in split_list_long:
                PROMPT_SUBLIST = prompt_dict[prompt_label]['prompt'](sub_list, LIST2)
                num_tokens_sublist = len(encoding.encode(PROMPT_SUBLIST))

                if num_tokens_sublist > max_tokens:
                    N += 1
                    all_prompts_valid = False
                    break
                else:
                    prompt_list.append(PROMPT_SUBLIST)

            if all_prompts_valid:
                break
            else:
                prompt_list = []  # Reset the prompt list for the next iteration

    print(f"Number of prompts: {len(prompt_list)}")

    return prompt_list

In [ ]:
def save_mapping(mapping_dictionary, file_name):

  path = file_name

  for key, value in mapping_dictionary.items():
    if value is None:
        mapping_dictionary[key] = []

  df_to_save = pd.DataFrame.from_dict(mapping_dictionary, orient='index').reset_index()
  df_to_save.to_excel(path + '.xlsx', index=False, engine='openpyxl')

  with open(path+'.json', 'w') as json_file:
    json.dump(mapping_dictionary, json_file, indent=4)

  return


In [ ]:
def transform_string(input_str):
    # Replacing single quotes that are not part of "'s" with double quotes
    #transformed_str = re.sub(r"(?<!s)'(?!s\b)", '"', input_str)
    transformed_str = re.sub(r"'(?!s\b)", '"', input_str)
    #transformed_str = re.sub(r"(?<!\S)'(?!s\b)", '"', input_str)

    return transformed_str

In [ ]:
def GPT_item_matches(LIST1, LIST2, mapping_name, temperature, max_tokens, starting_prompt, prompt_label):

  MODEL = "gpt-4"
  SYSTEM_ROLE = prompt_dict[prompt_label]['SYSTEM_ROLE']

  prompts = prep_prompt(LIST1, LIST2, max_tokens, prompt_label)

  ## load up the existing mapping. If no existing mapping exists, reinitialise it.
  try:
    with open(f'{mapping_name}.json', 'r') as json_file:
      data_dict_final = json.load(json_file)
  except:
    data_dict_final = {}
    for i in NDB_df['FoodDescription'].unique().tolist():
      data_dict_final[i] = None

  count=0

  for PROMPT in prompts[starting_prompt:]:

    response = openai.ChatCompletion.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": SYSTEM_ROLE},
          {"role": "user", "content": f"{PROMPT}"},
      ],
      temperature=temperature,
    )

    mapping_str_response = response['choices'][0]['message']['content']
    match = re.search(r"\{.*?\}", mapping_str_response, re.DOTALL)
    if match:
        data_str = match.group(0)
        try:
          data_dict = ast.literal_eval(data_str)
        except:
          data_str = transform_string(data_str)
          data_dict = ast.literal_eval(data_str)

        # keys in the mapping dictionary htat GPT4 was automatically renaming
        if 'Vegetable oil' in data_dict:
          data_dict['Vegetable oil (e.g. rapeseed)'] = data_dict.pop('Vegetable oil')

        if 'Onion pakora/bhaji' in data_dict:
          data_dict['Onion pakora/bhajia'] = data_dict.pop('Onion pakora/bhaji')

        if 'Chicken, breast, uncooked' in data_dict:
          data_dict['Chicken breast fillet, uncooked'] = data_dict.pop('Chicken, breast, uncooked')

        if 'Soya dessert (e.g. Alpro chocolate, caramel (including free from soya fromage frais))' in data_dict:
          data_dict['Soya dessert (e.g. Alpro chocolate, caramel (including free from soya fromage frais)'] = data_dict.pop('Soya dessert (e.g. Alpro chocolate, caramel (including free from soya fromage frais))')

        if 'Flavoured yoghurt drink (e.g. flavoured Kefir, Lassi, Pakeeza Passionfruit Lassi Thick Yogurt Drink 500g)' in data_dict:
          data_dict['Flavoured yoghurt drink (e.g. flavoured Kefir, Lassi, Yop)'] = data_dict.pop('Flavoured yoghurt drink (e.g. flavoured Kefir, Lassi, Pakeeza Passionfruit Lassi Thick Yogurt Drink 500g)')

        if 'Milky way (including supermarket brand equivalents)' in data_dict:
          data_dict['Milky way (including supermarket own brand equivalents)'] = data_dict.pop('Milky way (including supermarket brand equivalents)')

        if 'Sweet and sour pork, ready meal' in data_dict:
          data_dict['Sweet and sour pork, ready meal, with rice'] = data_dict.pop('Sweet and sour pork, ready meal')

        if 'Porridge sachet, syrup/fruit flavour, uncooked (e.g. Oat So Simple fruit flavours)' in data_dict:
          data_dict['Porridge sachet, syrup/fruit flavour, uncooked (e.g. Oat so Simple fruit flavours)'] = data_dict.pop('Porridge sachet, syrup/fruit flavour, uncooked (e.g. Oat So Simple fruit flavours)')

        try:
          for food_item, potential_matches in data_dict.items():
            if data_dict_final[food_item] is not None:
              potential_matches = potential_matches if potential_matches is not None else []
              union_list = list(set(data_dict_final[food_item]) | set(potential_matches))
              data_dict_final[food_item] = union_list
            else:
              data_dict_final[food_item] = potential_matches
        except:
          print(data_dict.keys())
          print(data_dict)
          break

          #print(food_item, data_dict[food_item])
          #print('\n')

        save_mapping(mapping_dictionary=data_dict_final, file_name=mapping_name)
        count+=1

        print(f'finished prompt {count}')

    else:
        print("GPT4 has not returned a dictionary. The response was: ")
        print(mapping_str_response)
        raise ValueError('No dictionary found!')

  return

In [ ]:
with open('NDB_CAT_TO_AISLE.json', 'r') as json_file:
    NDB_TO_AISLE = json.load(json_file)


In [ ]:
supplements_list = [
    "EVENING PRIMROSE OIL AND OTHER PLANT OILS",
    "FOLIC ACID",
    "IRON ONLY OR WITH VITAMIN C",
    "CALCIUM ONLY OR WITH VITAMIN D",
    "VITAMINS (TWO OR MORE INCL MULTIVITS) NO MINERALS",
    "MINERALS (TWO OR MORE INCL MULTIVITS & MINERALS)",
    "VITAMINS AND MINERALS (INCL MULTIVITS & MINERALS)",
    "NON-NUTRIENT SUPPLEMENTS (INCL HERBAL)",
    "OTHER NUTRIENT SUPPLEMENTS",
    "VITAMIN C ONLY",
    "SINGLE VITS OR MINS NOT Folic acid, Fe, Ca, Vit C",
    "COD LIVER OIL AND OTHER FISH OILS (INCL VIT A,D,E)",
    "MULTIVITAMIN AND/OR MINERALS WITH OMEGA 3"
]

In [ ]:
NDB_TO_AISLE_TO_MATCH = {key: value for key, value in NDB_TO_AISLE.items() if key not in supplements_list}

In [ ]:
NDB_matched_categories = ['SOFT DRINKS LOW CALORIE CARBONATED', 'SOFT DRINKS NOT LOW CALORIE CONCENTRATED', 'CANNED FRUIT IN JUICE',
                          'BEANS AND PULSES INCL READY MEAL & HOMEMADE DISHES', 'OTHER COOKING FATS AND OILS NOT PUFA',
                          'SAVOURY SAUCES PICKLES GRAVIES & CONDIMENTS', 'FRUIT JUICE', 'SOFT DRINKS LOW CALORIE RTD STILL',
                          'SOFT DRINKS NOT LOW CALORIE RTD STILL', 'APPLES AND PEARS NOT CANNED', 'OTHER FRUIT NOT CANNED', 'SALAD AND OTHER RAW VEGETABLES',
                          'OTHER VEGETABLES INCLUDING HOMEMADE DISHES', 'OTHER CEREALS', 'OTHER MEAT PRODUCTS MANUFACTURED INCL READY MEALS',
                          'MANUFACTURED COATED CHICKEN / TURKEY PRODUCTS', 'OTHER BEEF & VEAL INCLUDING HOMEMADE RECIPE DISHES', 'OTHER BACON AND HAM INCLUDING HOMEMADE DISHES',
                          'SANDWICHES', 'BISCUITS MANUFACTURED / RETAIL', 'BUNS CAKES & PASTRIES MANUFACTURED', 'OTHER MILK', 'CHOCOLATE CONFECTIONARY',
                          'OTHER LAMB INCLUDING HOMEMADE RECIPE DISHES', 'OTHER PORK INCLUDING HOMEMADE RECIPE DISHES',
                          'BURGERS AND KEBABS PURCHASED', 'MANUFACTURED BEEF PRODUCTS INCLUDING READY MEALS', 'OTHER CHICKEN / TURKEY INCL HOMEMADE RECIPE DISHES'
                          'MANUFACTURED CHICKEN PRODUCTS INCL READY MEALS', 'CHICKEN AND TURKEY DISHES', 'OTHER WHITE FISH INCLUDING HOMEMADE DISHES', 'WHITE FISH COATED OR FRIED',
                          'OTHER OILY FISH INCLUDING HOMEMADE DISHES','MANUFACTURED OILY FISH PRODUCTS INCL READY MEALS','OTHER SHELLFISH INCLUDING HOMEMADE DISHES', 'OTHER CHEESE',
                          'CREAM (INCLUDING IMITATION CREAM)', 'FROMAGE FRAIS AND DAIRY DESSERTS MANUFACTURED', 'YOGURT', 'ICE CREAM','OTHER LAMB INCLUDING HOMEMADE RECIPE DISHES',
                          'OTHER PORK INCLUDING HOMEMADE RECIPE DISHES',
                          'BURGERS AND KEBABS PURCHASED','MANUFACTURED BEEF PRODUCTS INCLUDING READY MEALS','OTHER MEAT PRODUCTS MANUFACTURED INCL READY MEALS',
                          'PIZZA', 'PASTA MANUFACTURED PRODUCTS & READY MEALS','OTHER PASTA INCLUDING HOMEMADE DISHES', 'HIGH FIBRE BREAKFAST CEREALS',
                                'OTHER BREAKFAST CEREALS (NOT HIGH FIBRE)', 'NUTS AND SEEDS', 'OTHER MANUFACTURED VEGETABLE PRODUCTS INCL RM', 'LEAFY GREEN VEGETABLES NOT RAW',
                                'OTHER POTATOES INCLUDING HOMEMADE DISHES', 'CANNED FRUIT IN SYRUP', 'PEAS NOT RAW', 'WHITE BREAD (NOT HIGH FIBRE; NOT MULTISEED BREAD)', 'WHOLEMEAL BREAD',
                          'BROWN GRANARY AND WHEATGERM BREAD', 'OTHER BREAD', 'CRISPS AND SAVOURY SNACKS', 'SUGAR CONFECTIONERY', 'SWEET SPREADS FILLINGS AND ICING','SUGAR', 'BUNS CAKES & PASTRIES HOMEMADE',
                                'OTHER FRIED / ROAST POTATOES INCL HOMEMADE DISHES', 'LIVER AND DISHES', 'MANUFACTURED MEAT PIES AND PASTRIES','COFFEE (MADE-UP WEIGHT)','SOFT DRINKS NOT LOW CALORIE CARBONATED',
                          'SWEET SPREADS FILLINGS AND ICING','SUGAR', 'BUNS CAKES & PASTRIES HOMEMADE'

                          ]

In [ ]:
categories_maybe_match = ['INFANT FORMULA', 'COMMERCIAL TODDLERS FOODS']

In [ ]:
# categories that don't need to be passed through gp4, a string-based match picks out good matches
categories_string_match = ['SAUSAGES', 'PASTA RICE AND OTHER CEREALS', 'BEEF VEAL AND DISHES', 'OTHER MILK AND CREAM', 'REDUCED FAT SPREAD (POLYUNSATURATED)',
                           'LOW FAT SPREAD NOT POLYUNSATURATED', 'SPONGE PUDDINGS - MANUFACTURED', 'BANANAS', 'CARROTS NOT RAW' ]

In [ ]:
NDB_TO_AISLE_TO_MATCH = {key: value for key, value in NDB_TO_AISLE_TO_MATCH.items() if key not in NDB_matched_categories}
NDB_TO_AISLE_TO_MATCH = {key: value for key, value in NDB_TO_AISLE_TO_MATCH.items() if key not in categories_maybe_match}
NDB_TO_AISLE_TO_MATCH = {key: value for key, value in NDB_TO_AISLE_TO_MATCH.items() if key not in categories_string_match}

In [ ]:
len(NDB_TO_AISLE_TO_MATCH.keys())

73

In [ ]:
def run_matching(ndb_category, starting_list, max_tokens, prompt_label, with_stop_words=False,
                 starting_prompt=0):

  start_time= time.time()

  ### set teh stop words
  #stop_words = ['veggie', 'vegetarian' ,'ice cream', 'vegan', 'yoghurt']
  #stop_words =['cake', 'crisps', 'snack', 'chicken', 'beef', 'pork', 'turkey', 'gammon','lamb', 'cheese' ,'ice cream', 'popcorn', 'bars', 'egg', 'roll', 'curry', 'bean', 'bread', 'vegetable']
  #stop_words = ['cake', 'crisps', 'sausage', 'beef', 'pork', 'gammon','lamb', 'vegetarian','vegan','steak','ice cream', 'popcorn', 'yoghurt', 'yogurt', 'mozzarella' ]

  #stop_words = ['crisps', 'sausage', 'beef', 'bacon', 'pork', 'gammon','lamb', 'chicken', 'turkey' ,'vegan' , 'egg' ,'ice cream', 'popcorn', 'yoghurt', 'yogurt' ]
  #stop_words = ['oil', 'crisps', 'sausage', 'beef', 'bacon', 'pork', 'gammon','lamb', 'chicken', 'turkey']

  stop_words = prompt_dict[prompt_label]['stop_words']

  possible_aisles = NDB_TO_AISLE_TO_MATCH[ndb_category]

  list_ndb = NDB_df[NDB_df['NDNS Food group']==ndb_category]['Local description'].unique().tolist()
  list_foodDB = fdb_df[fdb_df['aisle'].isin(possible_aisles)]['product_list_name'].unique().tolist()

  print(f'starting {ndb_category}')
  print(f'comparing with {len(list_foodDB)} items')

  if with_stop_words:
    list_foodDB = [i for i in list_foodDB if not any(sub in i.lower() for sub in stop_words)]

  print(f'Post stop words, comparing with {len(list_foodDB)} items')

  if len(list_ndb) > 150:
    print('Splitting into 6')
    ndb_item_sublist = split_list(list_ndb, N=6)
    for sublist in ndb_item_sublist[starting_list:]:
      GPT_item_matches(LIST1=list_foodDB, LIST2=sublist, mapping_name='NDB_to_FoodDB_GPT',
                       temperature=0, max_tokens=max_tokens, starting_prompt=starting_prompt, prompt_label=prompt_label)
      print(f'finished up to {sublist[-1]}')

  elif len(list_ndb) > 100:
    print('Splitting into 5')
    ndb_item_sublist = split_list(list_ndb, N=5)
    for sublist in ndb_item_sublist[starting_list:]:
      GPT_item_matches(LIST1=list_foodDB, LIST2=sublist, mapping_name='NDB_to_FoodDB_GPT', temperature=0,
                       max_tokens=max_tokens, starting_prompt=starting_prompt, prompt_label=prompt_label)
      print(f'finished up to {sublist[-1]}')

  elif len(list_ndb) > 30:
    print('Splitting into 2')
    ndb_item_sublist = split_list(list_ndb, N=2)
    for sublist in enumerate(ndb_item_sublist[starting_list:]):
      GPT_item_matches(LIST1=list_foodDB, LIST2=sublist, mapping_name='NDB_to_FoodDB_GPT', temperature=0,
                       max_tokens=max_tokens, starting_prompt=starting_prompt, prompt_label=prompt_label)

      print(f'finished up to {sublist[-1]}')

  else:
    GPT_item_matches(LIST1=list_foodDB, LIST2=list_ndb, mapping_name='NDB_to_FoodDB_GPT', temperature=0,
                     max_tokens=max_tokens, starting_prompt=starting_prompt, prompt_label=prompt_label)
    #print(f'finished up to {sublist[-1]}')

  end_time= time.time()

  return

In [ ]:
def run_matching_multiple_foodgroups(input_categories, prompt_label, starting_list,  max_tokens, starting_prompt=0):

  start_time= time.time()

  stop_words = prompt_dict[prompt_label]['stop_words']

  # !! assumes that the possibles aisles are the same for all food categories in the input list !! #
  possible_aisles = []

  for cat in input_categories:
    possible_aisle_i = NDB_TO_AISLE_TO_MATCH[cat]
    possible_aisles = list(set(possible_aisle_i) | set(possible_aisles))

  list_foodDB = fdb_df[fdb_df['aisle'].isin(possible_aisles)]['product_list_name'].unique().tolist()
  list_foodDB = [i for i in list_foodDB if not any(sub in i.lower() for sub in stop_words)]

  combined_ndb_list = []

  for cat in input_categories:
    list_ndb = NDB_df[NDB_df['NDNS Food group']==cat]['Local description'].unique().tolist()
    for i in list_ndb:
      combined_ndb_list.append(i)

  if len(combined_ndb_list) > 30:
    print('Splitting into 2')
    ndb_item_sublist = split_list(combined_ndb_list, N=2)
    for sublist in enumerate(ndb_item_sublist[starting_list:]):
      GPT_item_matches(LIST1=list_foodDB, LIST2=sublist, mapping_name='NDB_to_FoodDB_GPT', temperature=0,
                       max_tokens=max_tokens, starting_prompt=starting_prompt, prompt_label=prompt_label)

      print(f'finished up to {sublist[-1]}')

  else:
    GPT_item_matches(LIST1=list_foodDB, LIST2=combined_ndb_list, mapping_name='NDB_to_FoodDB_GPT', temperature=0,
                     max_tokens=max_tokens, starting_prompt=starting_prompt, prompt_label=prompt_label)

  end_time= time.time()

  return

## Run the GPT matching

In [ ]:
NDB_matched_categories_total = ['SOFT DRINKS LOW CALORIE CARBONATED', 'SOFT DRINKS NOT LOW CALORIE CONCENTRATED', 'CANNED FRUIT IN JUICE',
                          'BEANS AND PULSES INCL READY MEAL & HOMEMADE DISHES', 'OTHER COOKING FATS AND OILS NOT PUFA',
                          'SAVOURY SAUCES PICKLES GRAVIES & CONDIMENTS', 'FRUIT JUICE', 'SOFT DRINKS LOW CALORIE RTD STILL',
                          'SOFT DRINKS NOT LOW CALORIE RTD STILL', 'APPLES AND PEARS NOT CANNED', 'OTHER FRUIT NOT CANNED', 'SALAD AND OTHER RAW VEGETABLES',
                          'OTHER VEGETABLES INCLUDING HOMEMADE DISHES', 'OTHER CEREALS',

                                'OTHER MEAT PRODUCTS MANUFACTURED INCL READY MEALS',
                          'MANUFACTURED COATED CHICKEN / TURKEY PRODUCTS', 'OTHER BEEF & VEAL INCLUDING HOMEMADE RECIPE DISHES', 'OTHER BACON AND HAM INCLUDING HOMEMADE DISHES',
                          'SANDWICHES', 'BISCUITS MANUFACTURED / RETAIL', 'BUNS CAKES & PASTRIES MANUFACTURED', 'OTHER MILK', 'CHOCOLATE CONFECTIONARY',
                          'OTHER LAMB INCLUDING HOMEMADE RECIPE DISHES', 'OTHER PORK INCLUDING HOMEMADE RECIPE DISHES',
                          'BURGERS AND KEBABS PURCHASED', 'MANUFACTURED BEEF PRODUCTS INCLUDING READY MEALS', 'OTHER CHICKEN / TURKEY INCL HOMEMADE RECIPE DISHES'
                          'MANUFACTURED CHICKEN PRODUCTS INCL READY MEALS', 'CHICKEN AND TURKEY DISHES', 'OTHER WHITE FISH INCLUDING HOMEMADE DISHES', 'WHITE FISH COATED OR FRIED',
                          'OTHER OILY FISH INCLUDING HOMEMADE DISHES','MANUFACTURED OILY FISH PRODUCTS INCL READY MEALS','OTHER SHELLFISH INCLUDING HOMEMADE DISHES', 'OTHER CHEESE',
                          'CREAM (INCLUDING IMITATION CREAM)', 'FROMAGE FRAIS AND DAIRY DESSERTS MANUFACTURED', 'YOGURT', 'ICE CREAM','OTHER LAMB INCLUDING HOMEMADE RECIPE DISHES',
                          'OTHER PORK INCLUDING HOMEMADE RECIPE DISHES',
                          'BURGERS AND KEBABS PURCHASED','MANUFACTURED BEEF PRODUCTS INCLUDING READY MEALS','OTHER MEAT PRODUCTS MANUFACTURED INCL READY MEALS',
                          'PIZZA', 'PASTA MANUFACTURED PRODUCTS & READY MEALS','OTHER PASTA INCLUDING HOMEMADE DISHES', 'HIGH FIBRE BREAKFAST CEREALS',
                                'OTHER BREAKFAST CEREALS (NOT HIGH FIBRE)', 'NUTS AND SEEDS', 'OTHER MANUFACTURED VEGETABLE PRODUCTS INCL RM', 'LEAFY GREEN VEGETABLES NOT RAW',
                                'OTHER POTATOES INCLUDING HOMEMADE DISHES', 'CANNED FRUIT IN SYRUP', 'PEAS NOT RAW', 'WHITE BREAD (NOT HIGH FIBRE; NOT MULTISEED BREAD)', 'WHOLEMEAL BREAD',
                          'BROWN GRANARY AND WHEATGERM BREAD', 'OTHER BREAD', 'CRISPS AND SAVOURY SNACKS', 'SUGAR CONFECTIONERY', 'SWEET SPREADS FILLINGS AND ICING','SUGAR', 'BUNS CAKES & PASTRIES HOMEMADE',
                                'OTHER FRIED / ROAST POTATOES INCL HOMEMADE DISHES', 'LIVER AND DISHES', 'MANUFACTURED MEAT PIES AND PASTRIES','COFFEE (MADE-UP WEIGHT)','SOFT DRINKS NOT LOW CALORIE CARBONATED',
                          'SWEET SPREADS FILLINGS AND ICING','SUGAR', 'BUNS CAKES & PASTRIES HOMEMADE', 'WINE', 'BEER AND LAGER',"SOUP HOMEMADE","OTHER EGGS AND EGG DISHES INCLUDING HOMEMADE",  "SOUP MANUFACTURED/ RETAIL" ,
                                "TEA (MADE-UP WEIGHT)", "ALCOHOLIC SOFT DRINKS", 'HERBAL TEA (MADE-UP WEIGHT)', 'SPIRITS', 'MANUFACTURED EGG PRODUCTS INCLUDING READY MEALS', 'OTHER SAUSAGES INCLUDING HOMEMADE DISHES',
                                'MEAT ALTERNATIVES INCL READY MEALS & HOMEMADE DISH', 'MANUFACTURED PORK PRODUCTS INCLUDING READY MEALS', 'MANUFACTURED LAMB PRODUCTS INCLUDING READY MEALS', 'OTHER RICE INCLUDING HOMEMADE DISHES',
                                'RICE MANUFACTURED PRODUCTS & READY MEALS', 'MEAT ALTERNATIVES INCL READY MEALS & HOMEMADE DISH', 'TOMATOES RAW', 'LOW FAT SPREAD NOT POLYUNSATURATED',
                                'REDUCED FAT SPREAD (POLYUNSATURATED)', 'SPONGE PUDDINGS - MANUFACTURED', 'CARROTS NOT RAW', 'OTHER CEREAL BASED PUDDINGS - MANUFACTURED', 'OTHER MEAT INCLUDING HOMEMADE RECIPE DISHES', 'HOMEMADE MEAT PIES AND PASTRIES',
                                'READY MEALS / MEAL CENTRES BASED ON BACON AND HAM', 'MANUFACTURED PORK PRODUCTS INCLUDING READY MEALS', 'READY MEALS BASED ON SAUSAGES', 'BOTTLED WATER STILL OR CARBONATED',

                                'SOFT DRINKS NOT LOW CALORIE', 'SMOOTHIES 100% FRUIT AND/OR JUICE', 'LIQUEURS', 'FORTIFIED WINE',  'LOW ALCOHOL & ALCOHOL FREE BEER & LAGER', 'LOW ALCOHOL AND ALCOHOL FREE WINE', 'CIDER AND PERRY',
                                'NUTRITION POWDERS AND DRINKS',  'CHIPS PURCHASED INCLUDING TAKEAWAY', 'OTHER MANUFACTURED POTATO PRODUCTS FRIED/BAKED', 'BAKED BEANS', 'OTHER CEREAL BASED PUDDINGS - MANUFACTURED', 'CEREAL BASED MILK PUDDINGS - HOMEMADE', 'SPONGE PUDDINGS - HOMEMADE',
                                'CEREAL BASED MILK PUDDINGS - MANUFACTURED', 'BISCUITS HOMEMADE', 'OTHER CEREAL BASED PUDDINGS - HOMEMADE', 'FRUIT PIES HOMEMADE', 'CARROTS RAW', 'CITRUS FRUIT NOT CANNED'

                           'PRESERVES', 'SOFT DRINKS NOT LOW CALORIE', 'SMOOTHIES 100% FRUIT AND/OR JUICE', 'LIQUEURS', 'FORTIFIED WINE',  'LOW ALCOHOL & ALCOHOL FREE BEER & LAGER', 'LOW ALCOHOL AND ALCOHOL FREE WINE', 'CIDER AND PERRY',
                                'BOTTLED WATER STILL OR CARBONATED','NUTRITION POWDERS AND DRINKS' , 'POLYUNSATURATED OILS', 'ARTIFICIAL SWEETENERS', 'PRESERVES', 'CEREAL BASED MILK PUDDINGS - MANUFACTURED', 'BISCUITS HOMEMADE', 'OTHER CEREAL BASED PUDDINGS - HOMEMADE', 'FRUIT PIES HOMEMADE',
                                'CARROTS RAW', 'CITRUS FRUIT NOT CANNED', 'YOGURT FROMAGE FRAIS AND DAIRY DESSERTS', 'CHEDDAR CHEESE', 'COTTAGE CHEESE', 'WHOLE MILK', '1% Fat Milk', 'BLOCK MARGARINE', 'BUTTER', 'REDUCED FAT SPREAD (NOT POLYUNSATURATED)',
                                'MANUFACTURED SHELLFISH PRODUCTS INCL READY MEALS', 'MANUFACTURED WHITE FISH PRODUCTS INCL READY MEALS',
                     'MANUFACTURED CANNED TUNA PRODUCTS INCL READY MEALS', 'OTHER CANNED TUNA INCLUDING HOMEMADE DISHES', 'VEGETABLES NOT RAW',  'GREEN BEANS NOT RAW', 'TOMATOES NOT RAW', 'OTHER POTATO PRODUCTS & DISHES - MANUFACTURED',
                                'LOW ALCOHOL & ALCOHOL FREE CIDER & PERRY', 'COMMERCIAL TODDLERS DRINKS', 'FRUIT PIES MANUFACTURED', 'DAIRY DESSERTS HOMEMADE'

                          ]

In [ ]:
run_matching(ndb_category='COMMERCIAL TODDLERS DRINKS', starting_list=0, max_tokens=5000, prompt_label='drinks', with_stop_words=True,  starting_prompt=0)

starting COMMERCIAL TODDLERS DRINKS
comparing with 458 items
Post stop words, comparing with 458 items
starting token: 6604
Number of prompts: 2
finished prompt 1
finished prompt 2


In [ ]:
run_matching(ndb_category='LOW ALCOHOL & ALCOHOL FREE CIDER & PERRY', starting_list=0, max_tokens=4000, prompt_label='drinks', with_stop_words=True,  starting_prompt=0)


starting LOW ALCOHOL & ALCOHOL FREE CIDER & PERRY
comparing with 998 items
Post stop words, comparing with 998 items
starting token: 12536
Number of prompts: 4
finished prompt 1
finished prompt 2
finished prompt 3
finished prompt 4


In [ ]:
for cat in ['OTHER POTATO PRODUCTS & DISHES - MANUFACTURED']:
  run_matching(ndb_category=cat, starting_list=0, max_tokens=4000, prompt_label='default', with_stop_words=True,  starting_prompt=0)

starting OTHER POTATO PRODUCTS & DISHES - MANUFACTURED
comparing with 1535 items
Post stop words, comparing with 1535 items
starting token: 19397
Number of prompts: 7
finished prompt 1
finished prompt 2
finished prompt 3
finished prompt 4
finished prompt 5
finished prompt 6
finished prompt 7


In [ ]:
run_matching_multiple_foodgroups(input_categories=['FRUIT PIES MANUFACTURED', 'DAIRY DESSERTS HOMEMADE'], prompt_label='confectionary', starting_list=0,  max_tokens=4000, starting_prompt=0)

starting token: 43190
Number of prompts: 16
finished prompt 1
finished prompt 2
finished prompt 3
finished prompt 4
finished prompt 5
finished prompt 6
finished prompt 7
finished prompt 8
finished prompt 9
finished prompt 10
finished prompt 11
finished prompt 12
finished prompt 13
finished prompt 14
finished prompt 15
finished prompt 16


In [ ]:
for cat in ['FRUIT PIES MANUFACTURED', 'DAIRY DESSERTS HOMEMADE']:
  run_matching(ndb_category=cat, starting_list=0, max_tokens=4000, prompt_label='confectionary', with_stop_words=True,  starting_prompt=0)

In [ ]:
run_matching(ndb_category='TOMATOES NOT RAW', starting_list=0, max_tokens=4000, prompt_label='vegetables_fruit', with_stop_words=True,  starting_prompt=0)

starting TOMATOES NOT RAW
comparing with 2127 items
Post stop words, comparing with 2088 items
starting token: 24228
Number of prompts: 9
finished prompt 1
finished prompt 2
finished prompt 3
finished prompt 4
finished prompt 5
finished prompt 6
finished prompt 7
finished prompt 8
finished prompt 9
